# TrueRide Bicycles (TRB) — Sales, Customer & Product Analysis

**Goal:** turn 4 years of TRB transaction data (Q4 2020–Q1 2024) into a customer segmentation model (RFM),
a product performance view, and a profitability breakdown that leadership can act on.

**Structure of this notebook:**
1. Load & prepare data
2. Build the master transaction dataset
3. Exploratory check: commute distance vs. spending
4. RFM feature engineering
5. RFM scoring (1–5 scale per dimension)
6. Customer segmentation
7. Segment-level profiling
8. Customer demographics by segment
9. Export: customer-level segment table
10. Product performance analysis
11. Revenue & profit analysis
12. Price segmentation


## 1. Load & Prepare Data

Load the four source tables from the raw TRB Excel export. Keys are forced to `str` on load — they're identifiers, not numbers, so this avoids accidental numeric joins/sorting later.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Reference date used throughout for "days since last order" calculations.
# Matches the cutoff used in the original TRB dataset.
TODAY = pd.to_datetime('2024-12-30')

DATA_PATH = 'TRB Dataset.xlsx'  # update to your local path

In [ ]:
# Orders: force key columns to string so they behave as IDs, not numbers
order_dtypes = {
    'ProductKey': str,
    'CustomerKey': str,
    'PromotionKey': str
}
Order = pd.read_excel(DATA_PATH, sheet_name='Orders', dtype=order_dtypes)
Order.head()

In [ ]:
# Products
product_dtypes = {'ProductKey': str}
Product = pd.read_excel(DATA_PATH, sheet_name='Products', dtype=product_dtypes)
Product.dtypes

In [ ]:
# Customers
customer_dtypes = {'CustomerKey': str}
Customer = pd.read_excel(DATA_PATH, sheet_name='Customers', dtype=customer_dtypes)
Customer.head()

In [ ]:
# Promotions (not used in this version of the analysis — kept for completeness / future work)
Promotion = pd.read_excel(DATA_PATH, sheet_name='Promotion')
Promotion.head()

## 2. Build the Master Transaction Dataset

One row per order line, joined with customer and product attributes. This is the base table most of the analysis below builds from.

In [ ]:
master_df = (
    Order
    .merge(Customer, on='CustomerKey', how='left')
    .merge(Product, on='ProductKey', how='left')
)
master_df.shape

## 3. Exploratory Check — Commute Distance vs. Spending

Quick sanity check on whether customers who commute further tend to spend more (e.g. on higher-end bikes).
Not part of the core segmentation, but worth ruling in/out early.

In [ ]:
distance_map = {
    '0-1 Miles': 0,
    '1-2 Miles': 1,
    '2-5 Miles': 2,
    '5-10 Miles': 5,
    '10+ Miles': 10
}
master_df['CommuteDistanceNumeric'] = master_df['CommuteDistance'].map(distance_map)

commute_sales_corr = master_df['CommuteDistanceNumeric'].corr(master_df['SalesAmount'])
print(f"Correlation between commute distance and sales amount: {commute_sales_corr:.3f}")

**Result:** the correlation is weak — commute distance isn't a meaningful driver of spend at TRB. Not pursued further.

## 4. RFM Feature Engineering

Calculate the three raw RFM inputs per customer:
- **Recency** — days since their most recent order (lower = more recent = better)
- **Frequency** — total number of orders placed
- **Monetary** — total amount spent

TRB is a consumer-durable / high-ticket business (bikes are infrequent, high-value purchases), so Recency and
Monetary carry more segmentation weight than Frequency — that assumption shapes the segment rules in Section 6.

In [ ]:
rfm_raw = Order.groupby('CustomerKey').agg(
    Frequency=('ProductKey', 'count'),
    Recency=('OrderDate', lambda d: (TODAY - d.max()).days),
    Monetary=('SalesAmount', 'sum')
).reset_index()

rfm_raw.head()

## 5. RFM Scoring

Convert each raw RFM value into a 1–5 score using quintiles, so customers can be compared on a common scale
regardless of the underlying unit (days vs. order count vs. dollars).

- **Recency** is scored in reverse — the *most recent* customers (lowest day count) get the *highest* score (5).
- **Frequency** and **Monetary** are scored directly — higher values get higher scores.

In [ ]:
quintile_edges = [0, 0.2, 0.4, 0.6, 0.8, 1]

RFM = rfm_raw.copy()

# Recency: most recent customers score highest, so labels are reversed (5 -> 1)
RFM['R_score'] = pd.qcut(RFM['Recency'], q=quintile_edges, labels=[5, 4, 3, 2, 1], duplicates='drop').astype(int)

# Frequency & Monetary: highest values score highest (qcut labels start at 0, so add 1 to rebase to 1-5)
RFM['F_score'] = (pd.qcut(RFM['Frequency'], q=quintile_edges, labels=False, duplicates='drop') + 1).astype(int)
RFM['M_score'] = (pd.qcut(RFM['Monetary'], q=quintile_edges, labels=False, duplicates='drop') + 1).astype(int)

RFM['RFM_score'] = RFM['R_score'] + RFM['F_score'] + RFM['M_score']
RFM.head()

### Recency score distribution

Reference (from prior analysis run): the 5-scored band covers customers who ordered 318–410 days ago (~20.4% of
customers); the 1-scored band covers 626–1,464 days (~19.9%) — TRB's longest-inactive customers.

In [ ]:
r_score_dist = (
    RFM['R_score']
    .value_counts()
    .sort_index()
    .rename_axis('R_score')
    .reset_index(name='Count')
)
r_score_dist['Percentage'] = (r_score_dist['Count'] / r_score_dist['Count'].sum() * 100).round(2)
r_score_dist

### Frequency score distribution

Reference: score #1 (1–2 orders) covers ~41% of customers, score #4 (5–68 orders) covers ~18% — TRB's repeat/loyal
buyers are a clear minority of the base.

In [ ]:
f_score_dist = (
    RFM['F_score']
    .value_counts()
    .sort_index()
    .rename_axis('F_score')
    .reset_index(name='Count')
)
f_score_dist['Percentage'] = (f_score_dist['Count'] / f_score_dist['Count'].sum() * 100).round(2)
f_score_dist

## 6. Customer Segmentation

Segment definitions, weighted toward Recency and Monetary per the high-ticket/low-frequency assumption above:

| Segment | Rule | Logic |
|---|---|---|
| Most Valuable | R≥4, F≥3, M≥4 | Recent, frequent, and high-spending — top revenue contributors |
| Loyal | F=4 and R≥4 | Frequent, recent repeat buyers — high lifetime value even if per-order spend is lower |
| Big Spenders | M≥4 | High-value transactions regardless of frequency |
| New Customers | R=5 and F≤2 | Recent first-time or early-stage buyers |
| Churned Customer | R=1, F=1, M=1 | Lowest score on all three dimensions — effectively inactive |
| At Risk | R≤3, F≤2, M≤3 | Below-average across the board — a large group worth monitoring for further drop-off |
| Others | *(none of the above)* | Doesn't cleanly fit a named segment |

In [ ]:
def segment_customer(row):
    if row['R_score'] >= 4 and row['F_score'] >= 3 and row['M_score'] >= 4:
        return 'Most Valuable'
    elif row['F_score'] == 4 and row['R_score'] >= 4:
        return 'Loyal'
    elif row['M_score'] >= 4:
        return 'Big Spenders'
    elif row['R_score'] == 5 and row['F_score'] <= 2:
        return 'New Customers'
    elif row['R_score'] == 1 and row['F_score'] == 1 and row['M_score'] == 1:
        return 'Churned Customer'
    elif row['R_score'] <= 3 and row['F_score'] <= 2 and row['M_score'] <= 3:
        return 'At Risk'
    else:
        return 'Others'

RFM['Segment'] = RFM.apply(segment_customer, axis=1)
RFM['Segment'].value_counts()

## 7. Segment-Level Profiling

How many customers sit in each segment, how much they spend on average, and what share of total revenue they represent.

In [ ]:
segment_summary = RFM.groupby('Segment').agg(
    CustomerCount=('CustomerKey', 'count'),
    AverageSpending=('Monetary', 'mean'),
    TotalSpending=('Monetary', 'sum')
).round(2)

segment_summary['PctOfCustomers'] = (segment_summary['CustomerCount'] / segment_summary['CustomerCount'].sum() * 100).round(2)
segment_summary['PctOfTotalSpending'] = (segment_summary['TotalSpending'] / segment_summary['TotalSpending'].sum() * 100).round(2)
segment_summary['PctOfAvgSpending'] = (segment_summary['AverageSpending'] / segment_summary['AverageSpending'].sum() * 100).round(2)

segment_summary = segment_summary.sort_values('TotalSpending', ascending=False)
segment_summary['CumulativePctOfSpending'] = segment_summary['PctOfTotalSpending'].cumsum().round(2)

segment_summary

**Reading this table:**
- "Most Valuable" and "Big Spenders" together account for the large majority of total revenue despite being a
  minority of customers — TRB's revenue is concentrated, not evenly spread.
- "Loyal" customers repeat-purchase often but at lower order values, so their *average* spend looks unremarkable —
  their value shows up in resilience (see the README business narrative), not in per-order size.
- "At Risk" is a sizeable share of the customer base with non-trivial historical spend — a natural retention target
  before those customers fully churn.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sorted_summary = segment_summary.sort_values('PctOfTotalSpending', ascending=False)

bars = ax.bar(sorted_summary.index, sorted_summary['PctOfTotalSpending'], color=sns.color_palette('husl', len(sorted_summary)))
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.5, f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('% of Total Spending')
ax.set_title('Share of Total Revenue by Customer Segment')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. Customer Demographics

Age and income profile of the customer base, using the RFM segment assignments joined back to customer records.

In [ ]:
customer_profile = RFM.merge(Customer, on='CustomerKey', how='left')

customer_profile['BirthDate'] = pd.to_datetime(customer_profile['BirthDate'])
today = datetime.today()
customer_profile['Age'] = customer_profile['BirthDate'].apply(
    lambda dob: today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))
).astype(int)

customer_profile['Age'].describe()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x=customer_profile['Age'], color='olive')
plt.title('Distribution of Customer Age')
plt.xlabel('Age')
plt.show()

In [ ]:
customer_profile['YearlyIncome'].describe()

In [ ]:
plt.figure(figsize=(10, 8))
sns.boxplot(x=customer_profile['YearlyIncome'], color='gold')
plt.title('Distribution of Customer Yearly Income')
plt.xlabel('Yearly Income')
plt.show()

## 9. Export — Customer-Level Segment Table

Final customer-level table (RFM scores + segment + demographics), exported for use in Tableau.

In [ ]:
customer_profile.to_csv('Customer_analysis.csv', index=False)
customer_profile.head()

## 10. Product Performance Analysis

Which products sell the most overall, and which products each customer segment favours — this is what
connects the customer-level story back to the product/inventory decisions in the README recommendations.

In [ ]:
product_orders = Order.merge(Product, on='ProductKey', how='left')

product_by_segment = (
    product_orders
    .merge(customer_profile[['CustomerKey', 'Segment']], on='CustomerKey', how='left')
    [['ProductKey', 'OrderDate', 'CustomerKey', 'ProductName', 'ProductSubcategory',
      'ProductCategoryName', 'Segment', 'SalesAmount']]
)

# Drop exact duplicate transactions: same customer, same product, same order date
before = len(product_by_segment)
product_by_segment = product_by_segment.drop_duplicates(subset=['CustomerKey', 'ProductName', 'OrderDate'])
print(f"Removed {before - len(product_by_segment)} duplicate transaction rows")

product_by_segment.head()

In [ ]:
top_products_overall = (
    product_by_segment.groupby('ProductKey')['SalesAmount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
top_products_overall.head(10)

In [ ]:
top_products_by_segment = (
    product_by_segment.groupby(['Segment', 'ProductSubcategory'])['SalesAmount']
    .sum()
    .reset_index()
    .sort_values(['Segment', 'SalesAmount'], ascending=[True, False])
    .groupby('Segment')
    .head(10)
)
top_products_by_segment

In [ ]:
product_by_segment.to_csv('Product_by_segment.csv', index=False)

## 11. Revenue & Profit Analysis

Profit per order line, and a regional breakdown — this is the base data behind the "best sales year vs. best
profit year" and regional-gap findings in the README.

In [ ]:
Order['Profit'] = Order['SalesAmount'] - Order['ProductStandardCost'] - Order['TaxAmt']

regional_summary = Order.groupby('SalesTerritoryCountry').agg(
    OrderCount=('SalesOrderNumber', 'nunique'),
    TotalSales=('SalesAmount', 'sum'),
    TotalProfit=('Profit', 'sum')
).sort_values('TotalSales', ascending=False).round(2)

regional_summary

In [ ]:
tax_sales_corr = Order['SalesAmount'].corr(Order['TaxAmt'])
print(f"Correlation between sales amount and tax amount: {tax_sales_corr:.3f}")
# Expected to be strongly positive since tax is calculated as a function of sales amount — used as a data sanity check

In [ ]:
Order.to_csv('Order_with_profit.csv', index=False)

## 12. Price Segmentation

Bin products into price tiers to see where TRB's catalogue sits and which tier drives the most transactions.
Bin edges are based on the observed quartile breaks in `ListPrice`.

In [ ]:
price_view = Order.merge(Product, on='ProductKey', how='left')
price_view['ListPrice'].describe()

In [ ]:
plt.figure(figsize=(10, 8))
sns.boxplot(x=price_view['ListPrice'], color='orange')
plt.title('Distribution of Product List Price')
plt.xlabel('List Price')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(price_view['ListPrice'], bins=15, edgecolor='black')
plt.title('List Price Distribution')
plt.xlabel('List Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
price_view['PriceSegment'] = pd.cut(
    price_view['ListPrice'],
    bins=[0, 33, 590, 1300, float('inf')],
    labels=['Low-end', 'Mid-range', 'High-end', 'Premium']
)

price_view[['ProductKey', 'CustomerKey', 'ProductName', 'ListPrice', 'SalesAmount', 'PriceSegment']].head(20)